# Calculate VIMFC and MFC (850 hPa) - Model Only (Option 1)

This notebook computes moisture flux convergence using **Option 1 (model anomalies only)**:
- $\mathbf{VIMF} = \int (q' \mathbf{v}') rac{\Delta p}{g}$
- $	ext{VIMFC} = -
abla \cdot \mathbf{VIMF}$
- $\mathbf{MF}_{850} = q'_{850} \mathbf{v}'_{850}$
- $	ext{MFC}_{850} = -
abla \cdot \mathbf{MF}_{850}$

This version uses only `combined_ncep_r2_moist_lbm.nc` without basic state datasets.
Results are saved to `convergence_vimfc_850_model_only.nc`.


In [1]:
import os
from pathlib import Path
import numpy as np
import xarray as xr
import geocat.comp as gc

# Paths
NOTEBOOK_DIR = Path('/Users/rizzie/Work/PaperENSO/Scripts_v2/Supplement/ConvergenceModel')
LBM_PATH = Path('/Users/rizzie/Work/PaperENSO/Scripts_v2/Fig5/combined_ncep_r2_moist_lbm.nc')
OUTPUT_NC_PATH = NOTEBOOK_DIR / 'convergence_vimfc_850_model_only.nc'

print(f'LBM file exists: {LBM_PATH.exists()}')


LBM file exists: True


In [2]:
# Load model response dataset
ds_lbm = xr.open_dataset(LBM_PATH, decode_times=False)

# Constants & parameters
g = 9.80665                 # m s^-2
time_idx = 14               # 15th time step (Day 15 / t = 336.0 h)
lev_Pa = ds_lbm.lev * 100.0 # Convert hPa to Pa

# Layer thickness dp using reference standard surface pressure (1000 hPa + ps_prime)
ref_ps_Pa = 100000.0  # 1000 hPa in Pa

experiments = ['CTRL', 'EXP_B_S-123']

print(f'Selected experiments: {experiments}')
print(f'Selected time step: {time_idx} (t = {ds_lbm.time.values[time_idx]} h)')


Selected experiments: ['CTRL', 'EXP_B_S-123']
Selected time step: 14 (t = 336.0 h)


In [3]:
results = {}

for exp_id in experiments:
    # 1. Surface pressure and vertical layer thickness (dp)
    ps_prime = ds_lbm['p'].sel(experiment=exp_id).isel(time=time_idx).squeeze('lev_2')
    ps_total_Pa = ref_ps_Pa + ps_prime * 100.0  # reference 1000 hPa + perturbation
    
    dp_raw = gc.delta_pressure(lev_Pa, ps_total_Pa)
    dp = xr.DataArray(
        dp_raw,
        coords={'lat': ds_lbm.lat, 'lon': ds_lbm.lon, 'lev': lev_Pa},
        dims=['lat', 'lon', 'lev']
    )
    
    # 2. Extract perturbation wind and moisture fields
    u_prime = ds_lbm['u'].sel(experiment=exp_id).isel(time=time_idx).assign_coords(lev=lev_Pa)
    v_prime = ds_lbm['v'].sel(experiment=exp_id).isel(time=time_idx).assign_coords(lev=lev_Pa)
    q_prime = ds_lbm['q'].sel(experiment=exp_id).isel(time=time_idx).assign_coords(lev=lev_Pa)
    
    # 3. Vertically Integrated Moisture Flux (VIMF) and Convergence (VIMFC) - Model Only: q' * u'
    u_q = (q_prime * u_prime) * dp / g
    v_q = (q_prime * v_prime) * dp / g
    
    VIMF_u = u_q.sum(dim='lev')
    VIMF_v = v_q.sum(dim='lev')
    
    grads_u = gc.gradient(VIMF_u)
    grads_v = gc.gradient(VIMF_v)
    VIMFC = -(grads_u[1] + grads_v[0])
    VIMFC = xr.DataArray(VIMFC.values, coords=[('lat', ds_lbm.lat.values), ('lon', ds_lbm.lon.values)])
    
    # 4. Moisture Flux at 850 hPa (MF_850) and Convergence (MFC_850) - Model Only: q' * u'
    u_p_850 = u_prime.sel(lev=85000.0)
    v_p_850 = v_prime.sel(lev=85000.0)
    q_p_850 = q_prime.sel(lev=85000.0)
    
    MF_850_u = q_p_850 * u_p_850
    MF_850_v = q_p_850 * v_p_850
    
    grads_mf_u = gc.gradient(MF_850_u)
    grads_mf_v = gc.gradient(MF_850_v)
    MFC_850 = -(grads_mf_u[1] + grads_mf_v[0])
    MFC_850 = xr.DataArray(MFC_850.values, coords=[('lat', ds_lbm.lat.values), ('lon', ds_lbm.lon.values)])
    
    results[exp_id] = {
        'VIMF_u': VIMF_u,
        'VIMF_v': VIMF_v,
        'VIMFC': VIMFC,
        'MF_850_u': MF_850_u,
        'MF_850_v': MF_850_v,
        'MFC_850': MFC_850
    }
    print(f'Successfully calculated fields for {exp_id}')

# 5. Difference: EXP_B_S-123 - CTRL
diff_name = 'EXP_minus_CTRL'
results[diff_name] = {
    'VIMF_u': results['EXP_B_S-123']['VIMF_u'] - results['CTRL']['VIMF_u'],
    'VIMF_v': results['EXP_B_S-123']['VIMF_v'] - results['CTRL']['VIMF_v'],
    'VIMFC': results['EXP_B_S-123']['VIMFC'] - results['CTRL']['VIMFC'],
    'MF_850_u': results['EXP_B_S-123']['MF_850_u'] - results['CTRL']['MF_850_u'],
    'MF_850_v': results['EXP_B_S-123']['MF_850_v'] - results['CTRL']['MF_850_v'],
    'MFC_850': results['EXP_B_S-123']['MFC_850'] - results['CTRL']['MFC_850'],
}
print(f'Successfully calculated differences ({diff_name})')


Successfully calculated fields for CTRL


Successfully calculated fields for EXP_B_S-123
Successfully calculated differences (EXP_minus_CTRL)


In [4]:
# Build xarray Dataset
cases = ['CTRL', 'EXP_B_S-123', 'EXP_minus_CTRL']
case_coord = xr.DataArray(np.array(cases, dtype='S16'), dims='case', name='case')

def clean_da(da):
    return da.reset_coords(drop=True)

out_ds = xr.Dataset(
    {
        'VIMF_u': xr.concat([clean_da(results[c]['VIMF_u']) for c in cases], dim=case_coord),
        'VIMF_v': xr.concat([clean_da(results[c]['VIMF_v']) for c in cases], dim=case_coord),
        'VIMFC': xr.concat([clean_da(results[c]['VIMFC']) for c in cases], dim=case_coord),
        'MF_850_u': xr.concat([clean_da(results[c]['MF_850_u']) for c in cases], dim=case_coord),
        'MF_850_v': xr.concat([clean_da(results[c]['MF_850_v']) for c in cases], dim=case_coord),
        'MFC_850': xr.concat([clean_da(results[c]['MFC_850']) for c in cases], dim=case_coord),
    },
    coords={
        'lat': ds_lbm.lat.values,
        'lon': ds_lbm.lon.values,
    },
    attrs={
        'title': 'VIMFC and 850-hPa MFC (Model-Only Option 1: q prime * u prime)',
        'description': 'Computed from combined_ncep_r2_moist_lbm.nc without basic state',
        'time_index': str(time_idx),
        'time_hours': str(ds_lbm.time.values[time_idx]),
        'cases': ', '.join(cases),
    }
)

out_ds['VIMF_u'].attrs = {'long_name': 'Zonal vertically integrated moisture flux (model only)', 'units': 'kg m^-1 s^-1'}
out_ds['VIMF_v'].attrs = {'long_name': 'Meridional vertically integrated moisture flux (model only)', 'units': 'kg m^-1 s^-1'}
out_ds['VIMFC'].attrs = {'long_name': 'Vertically integrated moisture flux convergence (model only)', 'units': 'kg m^-2 s^-1'}
out_ds['MF_850_u'].attrs = {'long_name': 'Zonal moisture flux at 850 hPa (model only)', 'units': 'kg kg^-1 m s^-1'}
out_ds['MF_850_v'].attrs = {'long_name': 'Meridional moisture flux at 850 hPa (model only)', 'units': 'kg kg^-1 m s^-1'}
out_ds['MFC_850'].attrs = {'long_name': 'Moisture flux convergence at 850 hPa (model only)', 'units': 'kg kg^-1 s^-1'}

out_ds.to_netcdf(OUTPUT_NC_PATH)
print(f'Successfully saved results to {OUTPUT_NC_PATH}')
print(out_ds)


Successfully saved results to /Users/rizzie/Work/PaperENSO/Scripts_v2/Supplement/ConvergenceModel/convergence_vimfc_850_model_only.nc
<xarray.Dataset> Size: 247kB
Dimensions:   (lon: 64, lat: 32, case: 3)
Coordinates:
  * lon       (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4
  * lat       (lat) float64 256B 85.76 80.27 74.75 ... -74.75 -80.27 -85.76
  * case      (case) |S16 48B b'CTRL' b'EXP_B_S-123' b'EXP_minus_CTRL'
Data variables:
    VIMF_u    (case, lat, lon) float64 49kB -0.01047 -0.008405 ... 0.000719
    VIMF_v    (case, lat, lon) float64 49kB 0.01396 0.01406 ... -0.001468
    VIMFC     (case, lat, lon) float64 49kB -2.142e-08 -1.522e-08 ... -1.563e-08
    MF_850_u  (case, lat, lon) float32 25kB -2.859e-06 -2.173e-06 ... nan nan
    MF_850_v  (case, lat, lon) float32 25kB 4.706e-06 4.637e-06 ... nan nan
    MFC_850   (case, lat, lon) float64 49kB 8.16e-12 8.742e-12 ... nan nan
Attributes:
    title:        VIMFC and 850-hPa MFC (Model-Only Option 1: q prime 